# Lesson 5.3 · KNN in practice

With scaling in place and few useful features KNN is a surprisingly strong baseline. It has, however, three characteristics to know: prediction cost, sensitivity to the choice of `k` and to the size of the data, and
a use that goes beyond classification (regression, imputation of missing values).

**What you will learn**

- How much `predict` costs and how the search algorithms (`brute`, `kd_tree`, `ball_tree`) change things.
- How `k` is chosen and how regression with neighbors works.
- `KNNImputer` as an alternative to imputation with the mean.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import time
from sklearn.model_selection import StratifiedKFold, validation_curve
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification

## The cost is all in `predict`

`fit` is almost free, `predict` computes the distance from every training point. With `brute` the cost is $O(n\cdot d)$ for every prediction. With `kd_tree` and `ball_tree` scikit-learn organizes the data in a
tree to skip most of the comparisons: very effective with **few dimensions** (up to about 10-15), of little use with many.

In [ ]:
rng = np.random.default_rng(0)
for d in [3, 30]:
    Xb = rng.normal(size=(30000, d)); Xq = rng.normal(size=(1000, d))
    row = []
    for alg in ["brute", "kd_tree", "ball_tree"]:
        m = KNeighborsClassifier(5, algorithm=alg).fit(Xb, rng.integers(0, 2, len(Xb)))
        t = time.time(); m.predict(Xq); row.append(f"{alg} {time.time() - t:.2f}s")
    print(f"{d:2d} dimensions, 30000 training examples, 1000 predictions: " + " | ".join(row))

With 3 dimensions the tree is much faster than the exhaustive computation (about 5 times); with 30 it becomes **a disadvantage** (over twenty times slower: building and visiting the tree costs more than the direct comparison). The times depend on your machine: look at the ratios, not at the seconds. The default (`algorithm="auto"`) chooses for you.
It is also the reason why a KNN in production with millions of examples becomes a problem of **approximate search** (ANN indexes), not of scikit-learn.

## Choosing `k`

It is chosen with cross-validation like any hyperparameter, looking at the whole curve and not only at the maximum (D1 of module 3). As a starting point the rule of thumb $k \approx \sqrt{n}$ holds, then you adjust.

In [ ]:
Xc, yc = make_classification(1500, n_features=8, n_informative=4, n_redundant=0, class_sep=0.8, random_state=0)
ks = [1, 3, 5, 9, 15, 25, 41, 61, 91]
train, val = validation_curve(make_pipeline(StandardScaler(), KNeighborsClassifier()), Xc, yc, param_name="kneighborsclassifier__n_neighbors", param_range=ks,
                              cv=StratifiedKFold(5, shuffle=True, random_state=0))
pd.DataFrame({"train": train.mean(1), "validation": val.mean(1)}, index=ks).round(3).T

The training score collapses as `k` grows while the validation one goes up, touches a maximum and then goes down: it is the U-shaped curve of complexity. Note that the maximum is **flat**: many values of `k` are equivalent
within the uncertainty of the estimate; when in doubt you choose the simplest (larger `k`, smoother model).

## Regression with neighbors

`KNeighborsRegressor` predicts the **mean** of the target of the `k` neighbors. It does not extrapolate (outside the training range the prediction stays flat) but it captures nonlinear relationships without writing any function.

In [ ]:
x = np.sort(rng.uniform(0, 6, 120)); yr = np.sin(x) + rng.normal(0, 0.25, 120)
grid = np.linspace(-1, 7, 300).reshape(-1, 1)
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.scatter(x, yr, s=12, color="gray")
for k in [1, 10, 40]:
    ax.plot(grid, KNeighborsRegressor(k).fit(x.reshape(-1, 1), yr).predict(grid), label=f"k = {k}")
ax.legend(); plt.show()

With `k = 1` the curve "jumps" from one point to the next (it follows the noise); with `k = 40` it is too smooth; outside [0, 6] the prediction is constant: **KNN does not extrapolate**.

## KNNImputer: neighbors to fill the missing values

The same idea for missing values: `KNNImputer` fills a missing value with the mean of the `k` neighbors (computed on the available columns). It requires **scaled** features and works well when the columns are correlated with each other:
you will see it in exercise B3.

> **Common mistake.** Using KNN because "it is simple" on a dataset with hundreds of features and millions of rows: it is not simple to put in production, and in high dimension it is almost always beaten by a linear model or by trees.

## Check your understanding

> *Click on a question to show its answer.*

<details><summary>When does KNN make sense?</summary>

As a quick baseline to try, with few scaled and meaningful features, data that is not huge and irregular boundaries; in recommendation and retrieval systems (with approximate indexes). It does not make sense with many irrelevant features, huge data to serve with low latency, or when you need to extrapolate.

</details>

<details><summary>How do you choose between `kd_tree` and `brute`?</summary>

Usually you don't: `algorithm="auto"` decides. In general trees help with few dimensions and many examples; with many dimensions (over 15-20) or few examples the exhaustive computation is just as fast.

</details>